In [ ]:
# !pip install -U langchain
# !pip install -U langchain-cohere
# !pip install -U langchain-community
# !pip install -U langchain-text-splitters
# !pip install -U langchain-chroma
# !pip install -U pypdfA
# !pip install -U chromadb
# !pip install -U gradio

In [ ]:
# !pip install pypdf

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from google.colab import userdata
from google.colab import files
from langchain_cohere import ChatCohere
from langchain_cohere import CohereEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
key = userdata.get('coherekey')

In [ ]:
uploaded = files.upload()

pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
print("Uploaded PDFs:")
for p in pdf_paths:
    print(" -", p)

Saving [ML&PR 2026] Lec0 Intro.pdf to [ML&PR 2026] Lec0 Intro.pdf
Saving [ML&PR 2026] Lec1 Regression I.pdf to [ML&PR 2026] Lec1 Regression I.pdf
Saving [ML&PR 2026] Lec2 Regression II.pdf to [ML&PR 2026] Lec2 Regression II.pdf
Uploaded PDFs:
 - [ML&PR 2026] Lec0 Intro.pdf
 - [ML&PR 2026] Lec1 Regression I.pdf
 - [ML&PR 2026] Lec2 Regression II.pdf


In [ ]:
documents = []

for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs = loader.load()
    for d in docs:
        d.metadata["source_file"] = path
    documents.extend(docs)
    print(f"{path}: {len(docs)} pages loaded")

print("\nTotal pages loaded across all PDFs:", len(documents))

[ML&PR 2026] Lec0 Intro.pdf: 18 pages loaded
[ML&PR 2026] Lec1 Regression I.pdf: 27 pages loaded
[ML&PR 2026] Lec2 Regression II.pdf: 19 pages loaded

Total pages loaded across all PDFs: 64


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))

Total chunks: 63


In [ ]:
embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="multi_pdf_rag"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [ ]:
llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about a set of uploaded PDF documents.

Answer the user's question using ONLY the context below.
If the answer cannot be found in the context, say "I don't know." Do not invent information.

Context:
{context}

Question:
{question}
""")


In [ ]:
def format_docs_with_sources(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source_file", "unknown file")
        page = doc.metadata.get("page", None)
        page_label = f"page {page + 1}" if isinstance(page, int) else "page unknown"
        formatted.append(f"[{source} — {page_label}]\n{doc.page_content}")
    return "\n\n".join(formatted)


def get_sources(docs):
    seen = []
    for doc in docs:
        source = doc.metadata.get("source_file", "unknown file")
        page = doc.metadata.get("page", None)
        page_label = f"p.{page + 1}" if isinstance(page, int) else "p.?"
        label = f"{source} ({page_label})"
        if label not in seen:
            seen.append(label)
    return seen


def ask(question: str, active_retriever=None):
    active_retriever = active_retriever or retriever
    docs = active_retriever.invoke(question)
    context = format_docs_with_sources(docs)
    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    sources = get_sources(docs)
    return response.content, sources

In [ ]:
import os
import chromadb
import gradio as gr

CHROMA_COLLECTION_NAME = "multi_pdf_rag_gradio"
chroma_client = chromadb.EphemeralClient()

state = {"retriever": None}


def build_index(files):
    try:
        chroma_client.delete_collection(CHROMA_COLLECTION_NAME)
    except Exception:
        pass

    if not files:
        state["retriever"] = None
        return "No PDFs uploaded yet."

    docs = []
    for f in files:
        path = f.name if hasattr(f, "name") else f
        loader = PyPDFLoader(path)
        loaded = loader.load()
        for d in loaded:
            d.metadata["source_file"] = os.path.basename(path)
        docs.extend(loaded)

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    doc_chunks = splitter.split_documents(docs)

    vs = Chroma.from_documents(
        documents=doc_chunks,
        embedding=embeddings,
        collection_name=CHROMA_COLLECTION_NAME,
        client=chroma_client
    )
    state["retriever"] = vs.as_retriever(search_kwargs={"k": 5})

    names = ", ".join(os.path.basename(f.name if hasattr(f, "name") else f) for f in files)
    return f"✅ Loaded {len(files)} PDF(s) ({names}) \u2014 {len(doc_chunks)} chunks. You can start asking questions."


def chat_fn(message, history):
    if state["retriever"] is None:
        return "Please upload at least one PDF in the sidebar first \u2014 it will be indexed automatically."


    answer, sources = ask(message, state["retriever"])
    if sources:
        answer += "\n\n\U0001F4CE **Sources:** " + "; ".join(sources)
    return answer


with gr.Blocks(title="Multi-PDF Chatbot") as demo:
    gr.Markdown("# \U0001F4DA Multi-PDF RAG Chatbot")

    with gr.Sidebar(position="right"):
        gr.Markdown("### Upload PDFs")
        file_upload = gr.Files(label="Upload PDFs", file_types=[".pdf"], file_count="multiple")
        status = gr.Markdown("No PDFs uploaded yet.")

    # Chat lives in the main (left) area.
    gr.ChatInterface(
        fn=chat_fn,
        title=None,
        description="Ask a question about any of the uploaded PDFs, or one that spans several of them.",
        chatbot=gr.Chatbot(height=350)
    )

    file_upload.change(fn=build_index, inputs=file_upload, outputs=status)

In [ ]:
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://580b8ea2fece7ac683.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
